[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/quickstarts/python_quickstart.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">Python 快速入门指南</h1>

本 Notebook 汇集所有 Python 快速入门指南中的代码，可在此查看：

🔹 [追踪入门](https://arize.com/docs/phoenix/get-started/get-started-tracing)

🔹 [评估入门](https://arize.com/docs/phoenix/get-started/get-started-evaluations)

🔹 [提示词入门](https://arize.com/docs/phoenix/get-started/get-started-prompt-playground)

🔹 [实验入门](https://arize.com/docs/phoenix/get-started/get-started-datasets-and-experiments)

本 Notebook 将使用 CrewAI 框架创建多智能体系统，收集追踪、运行评估、迭代智能体并开展实验。构建的是金融分析与研究聊天机器人，输入为股票代码和关注重点，输出为包含相关最新研究的金融报告。



### 追踪入门

In [ ]:
%pip install -qqqqq arize-phoenix crewai crewai-tools openinference-instrumentation-crewai openinference-instrumentation-openai openai

In [ ]:
import os
from getpass import getpass

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")

if not (serper_api_key := os.getenv("SERPER_API_KEY")):
    serper_api_key = getpass("🔑 请输入 Serper API 密钥： ")

# 指向 Phoenix 实例。本地 `uvx arize-phoenix serve` 监听
# http://localhost:6006，无需 API 密钥。按回车接受默认值；
# 若该地址没有服务监听（例如 Colab 等托管环境），Phoenix 会
# 在本 Notebook 中启动。
if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = (
        input("🦅 Phoenix 端点（按回车使用 http://localhost:6006）： ").strip()
        or "http://localhost:6006"
    )

# 仅在 Phoenix 实例启用身份验证时需要 API 密钥。
phoenix_api_key = os.getenv("PHOENIX_API_KEY")
if not phoenix_api_key and phoenix_endpoint != "http://localhost:6006":
    phoenix_api_key = getpass("🔑 Phoenix API 密钥（未启用身份验证时按回车跳过）： ")

if phoenix_endpoint == "http://localhost:6006":
    from urllib.request import urlopen

    try:
        urlopen(phoenix_endpoint, timeout=5)
    except Exception:
        import phoenix as px

        px.launch_app()

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint
if phoenix_api_key:
    os.environ["PHOENIX_API_KEY"] = phoenix_api_key
os.environ["SERPER_API_KEY"] = serper_api_key
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="crewai-tracing-quickstart", auto_instrument=True)

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()

researcher = Agent(
    role="Financial Research Analyst",
    goal="收集目标公司或市场的最新金融数据、趋势与新闻",
    backstory="""
        你是资深金融研究分析师。
    """,
    verbose=True,
    allow_delegation=False,
    max_iter=2,
    tools=[search_tool],
)

writer = Agent(
    role="Financial Report Writer",
    goal="整理与总结金融研究，形成清晰、可操作的洞察",
    backstory="""
        你是经验丰富的金融内容作者。
    """,
    verbose=True,
    allow_delegation=True,
    max_iter=1,
)

task1 = Task(
    description="""
        研究对象：{tickers}
        重点：{focus}
    """,
    expected_output="包含网络搜索发现的详细金融研究摘要",
    agent=researcher,
)

task2 = Task(
    description="根据以上研究撰写报告。",
    expected_output="完善的金融分析报告",
    agent=writer,
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    verbose=True,
    process=Process.sequential,
)

In [ ]:
user_inputs = {"tickers": "TSLA", "focus": "financial analysis and market outlook"}

result = crew.kickoff(inputs=user_inputs)

### 评估入门

In [ ]:
test_queries = [
    {"tickers": "AAPL", "focus": "financial analysis and market outlook"},
    {"tickers": "NVDA", "focus": "valuation metrics and growth prospects"},
    {"tickers": "AMZN", "focus": "profitability and market share"},
    {"tickers": "AAPL, MSFT", "focus": "comparative financial analysis"},
    {"tickers": "META, SNAP, PINS", "focus": "social media sector trends"},
    {"tickers": "RIVN", "focus": "financial health and viability"},
    {"tickers": "SNOW", "focus": "revenue growth trajectory"},
    {"tickers": "KO", "focus": "dividend yield and stability"},
    {"tickers": "META", "focus": "latest developments and stock performance"},
    {"tickers": "AAPL, MSFT, GOOGL, AMZN, META", "focus": "big tech comparison and market outlook"},
    {"tickers": "AMC", "focus": "financial analysis and market sentiment"},
]

In [ ]:
for query in test_queries:
    crew.kickoff(inputs=query)

In [ ]:
financial_completeness_template = """
你正在评估金融研究报告是否以全面覆盖的方式正确完成用户任务的所有部分。

用户输入：{attributes.input.value}

生成的报告：
{attributes.output.value}

要标记为 "complete"，报告必须满足以下全部严格要求：

1. 股票代码覆盖（必需）：
   - 涵盖输入提到的所有公司或股票代码
   - 若有多个代码，每个都必须有专门分析，不能只是顺带提及
   - 适用时，多个代码必须提供比较分析

2. 关注领域覆盖（必需）：
   - 回应输入提到的全部关注领域
   - 若关注多个主题，例如“收益和展望”，两者都必须充分讨论
   - 每个关注领域必须有实质内容，不能只是简略提及

3. 财务数据要求（必需）：
   - 每个股票代码都必须包含：
     * 当前或近期股价或表现数据
     * 至少 2 个关键财务比率，例如 P/E、P/B、债务权益比、ROE
     * 收入或盈利信息
     * 最近 6 个月内的新闻或动态
   - 若指定具体指标，例如 P/E，必须明确提供

4. 深度要求（必需）：
   - 每个股票代码至少有 3–4 句专门分析
   - 没有具体数据的泛泛陈述不算
   - 报告必须体现充分研究，而非表面覆盖

5. 比较要求（多个股票代码时）：
   - 若请求至少 2 个股票代码，必须包含直接比较
   - 比较应并列涵盖多个关键指标
   - “两家公司都很好”等泛泛陈述不满足要求
   - 必须明确说明具体指标上哪家公司更好或更差

不满足以上任一要求，报告即为 "incomplete"，包括：
- 遗漏任一股票代码，或只简略提及
- 未回应任一关注领域，或只作表面讨论
- 任一股票代码缺少所需财务数据
- 分析泛泛，没有具体指标或数据
- 请求多个股票代码却未提供比较
- 任一股票代码不满足深度要求

先仅回复一个词："complete" 或 "incomplete"
然后详细解释具体满足或未满足的要求。
"""

In [ ]:
from phoenix.client import Client

px_client = Client()
df = px_client.spans.get_spans_dataframe(project_name="crewai-tracing-quickstart")
parent_spans = df[df["span_kind"] == "CHAIN"]

In [ ]:
from phoenix.evals import LLM

llm = LLM(model="gpt-4o", provider="openai")

In [ ]:
from phoenix.evals import ClassificationEvaluator

completeness_evaluator = ClassificationEvaluator(
    name="completeness",
    prompt_template=financial_completeness_template,
    llm=llm,
    choices={"complete": 1.0, "incomplete": 0.0},
)

In [ ]:
from phoenix.evals import async_evaluate_dataframe

from phoenix.trace import suppress_tracing

with suppress_tracing():
    results_df = await async_evaluate_dataframe(
        dataframe=parent_spans, evaluators=[completeness_evaluator], concurrency=10
    )

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

evaluations = to_annotation_dataframe(dataframe=results_df)

Client().spans.log_span_annotations_dataframe(dataframe=evaluations)

### 实验入门

In [ ]:
researcher = Agent(
    role="Financial Research Analyst",
    goal="""收集目标公司或市场的最新金融数据、趋势与新闻。
        确保包含至少 2 个财务比率（例如 P/E 或 P/B）、最近 6 个月新闻，以及当前股价或表现数据。""",
    backstory="""
            你是资深金融研究分析师。
        """,
    verbose=True,
    allow_delegation=False,
    max_iter=2,
    tools=[search_tool],
)
writer = Agent(
    role="Financial Report Writer",
    goal="整理与总结金融研究，形成清晰、可操作的洞察. 若有多个股票代码，务必包含专门的比较章节。",
    backstory="""
        你是经验丰富的金融内容作者。
    """,
    verbose=True,
    allow_delegation=True,
    max_iter=1,
)
updated_crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    verbose=True,
    process=Process.sequential,
)

In [ ]:
def my_task(example):
    result = updated_crew.kickoff(inputs=example.input)
    return result

In [ ]:
from phoenix.evals import ClassificationEvaluator

completeness_evaluator = ClassificationEvaluator(
    name="completeness",
    prompt_template=financial_completeness_template,
    llm=llm,
    choices={"complete": 1.0, "incomplete": 0.0},
)

completeness_evaluator.bind(
    {
        "attributes.input.value": "input",
        "attributes.output.value": "output",
    }
)

evaluators = [completeness_evaluator]

**运行下一个单元前**，需要在 Phoenix 界面中根据失败的追踪创建数据集：

1. 打开 Phoenix 实例中的 **crewai-tracing-quickstart** 项目。
2. 点击 **Traces** 标签页，在筛选栏输入：`evals['completeness'].label == 'incomplete'`
3. 选择所有筛选后的追踪，点击 **Add to Dataset**，命名为 **"python quickstart fails"**。

此数据集保存智能体处理不佳的输入，可通过改进后的智能体重新运行，衡量变更是否真正有效。

In [ ]:
client = Client()
dataset = client.datasets.get_dataset(dataset="python quickstart fails")

In [ ]:
experiment = client.experiments.run_experiment(dataset=dataset, task=my_task, evaluators=evaluators)